# WQ Alpha India — Research Walkthrough

End-to-end demonstration of the `wqalpha` framework:

1. Load NSE equity panel data  
2. Compute all 101 WorldQuant alpha signals  
3. Evaluate predictive power (IC, ICIR, hit ratio)  
4. Backtest a long-short portfolio  
5. Risk attribution (CAPM, sector neutralization)  
6. Visualize results  

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Use a clean style
plt.rcParams.update({
    'figure.facecolor': '#1A1A2E',
    'axes.facecolor': '#1A1A2E',
    'axes.edgecolor': '#333355',
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'grid.color': '#333355',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
})

print('Libraries loaded successfully.')

## 1. Load Data

In [ ]:
from wqalpha.data import DataLoader, wide

panel = DataLoader.from_csv('../data/india_equities_sample.csv').load()

n_dates = panel.index.get_level_values('date').nunique()
n_symbols = panel.index.get_level_values('symbol').nunique()
print(f'Panel: {len(panel):,} rows | {n_symbols} symbols | {n_dates} dates')
print(f'Date range: {panel.index.get_level_values("date").min().date()} to '
      f'{panel.index.get_level_values("date").max().date()}')
panel.head()

In [ ]:
# Returns matrix (date x symbol)
returns = wide(panel, 'returns')
close = wide(panel, 'close')
volume = wide(panel, 'volume')

print('Returns matrix shape:', returns.shape)
print(f'Mean daily return: {returns.stack().mean():.4%}')
print(f'Mean daily vol:    {returns.stack().std():.4%}')

## 2. Compute All 101 Alpha Signals

In [ ]:
from wqalpha.alphas import register_all_alphas
from wqalpha.registry import AlphaRegistry

registry = AlphaRegistry()
register_all_alphas(registry)
print(f'Registered {len(registry.names())} alphas.')

# Compute a single alpha to verify
alpha_001 = registry.compute('alpha_001', panel)
print('Alpha 001 sample values:')
print(alpha_001.dropna().describe())

In [ ]:
# Compute all 101 alphas (takes ~30-60s)
print('Computing all 101 alphas...')
all_alphas = registry.compute_all(panel)
print(f'All alphas shape: {all_alphas.shape}')
print(f'Coverage (non-NaN %): {all_alphas.notna().mean().mean():.1%}')

## 3. IC / ICIR Analysis

In [ ]:
from wqalpha.metrics import forward_returns, information_coefficient, icir, hit_ratio

fwd1 = forward_returns(returns, horizon=1)

# Compute IC for all 101 alphas
ic_results = {}
for name in registry.names():
    sig = all_alphas[name].unstack('symbol').reindex(returns.index)
    ic_series = information_coefficient(sig, fwd1)
    ic_results[name] = {
        'mean_ic': ic_series.dropna().mean(),
        'icir': icir(ic_series),
        'hit_ratio': hit_ratio(ic_series),
    }

ic_df = pd.DataFrame(ic_results).T
ic_df = ic_df.sort_values('icir', ascending=False)
print('Top 15 alphas by ICIR:')
print(ic_df.head(15).to_string(float_format='{:.4f}'.format))

In [ ]:
# Plot ICIR distribution across all 101 alphas
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(ic_df['mean_ic'].dropna(), bins=20, color='#4C9BE8', alpha=0.75)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Mean IC Distribution', color='white')
axes[0].set_xlabel('Mean IC', color='white')

axes[1].hist(ic_df['icir'].dropna(), bins=20, color='#9BE84C', alpha=0.75)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('ICIR Distribution', color='white')
axes[1].set_xlabel('ICIR', color='white')

axes[2].hist(ic_df['hit_ratio'].dropna(), bins=20, color='#E84C9B', alpha=0.75)
axes[2].axvline(0.5, color='red', linestyle='--')
axes[2].set_title('Hit Ratio Distribution', color='white')
axes[2].set_xlabel('Hit Ratio', color='white')

plt.suptitle('Alpha Quality Metrics — All 101 WQ Alphas', fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

## 4. Deep-Dive: Best Alpha

In [ ]:
from wqalpha.metrics import alpha_decay
from wqalpha.visualization import (
    plot_ic_distribution, plot_ic_timeseries, plot_alpha_decay
)

# Pick the alpha with the highest ICIR
best_name = ic_df['icir'].idxmax()
print(f'Best alpha by ICIR: {best_name}')
print(ic_df.loc[best_name])

best_sig = all_alphas[best_name].unstack('symbol').reindex(returns.index)
best_ic = information_coefficient(best_sig, fwd1)

fig = plot_ic_distribution(best_ic, alpha_name=best_name)
plt.show()

fig = plot_ic_timeseries(best_ic, alpha_name=best_name)
plt.show()

In [ ]:
decay = alpha_decay(best_sig, returns, max_lag=20)
fig = plot_alpha_decay(decay, alpha_name=best_name)
plt.show()

## 5. Long-Short Backtest

In [ ]:
from wqalpha.backtest import long_short_weights, portfolio_returns, performance_stats
from wqalpha.visualization import plot_cumulative_returns, plot_drawdown, plot_quantile_returns

weights = long_short_weights(best_sig, long_quantile=0.90, short_quantile=0.10)
strategy_returns = portfolio_returns(weights, returns, transaction_cost_bps=10)

stats = performance_stats(strategy_returns)
print('=== Performance Statistics ===')
for k, v in stats.items():
    if k in ('sharpe', 'sortino', 'cagr', 'volatility', 'max_drawdown'):
        print(f'  {k:20s}: {v:.4f}')

In [ ]:
benchmark = returns.mean(axis=1).rename('EW Index')

fig = plot_cumulative_returns(strategy_returns, benchmark=benchmark, label=best_name)
plt.show()

fig = plot_drawdown(strategy_returns, label=best_name)
plt.show()

In [ ]:
fig = plot_quantile_returns(best_sig, returns, n_quantiles=5, periods=1)
plt.show()

## 6. Risk Attribution

In [ ]:
from wqalpha.risk import capm, sector_neutralize
from wqalpha.visualization import plot_factor_exposure

market_returns = returns.mean(axis=1)
capm_result = capm(strategy_returns, market_returns)
print('CAPM Results:')
print(capm_result)

fig = plot_factor_exposure(capm_result, title=f'CAPM Factor Exposure — {best_name}')
plt.show()

In [ ]:
# Sector-neutralized version
sector_map = panel['sector'].groupby(level='symbol').last()
sig_neutral = sector_neutralize(best_sig, sector_map)

w_neutral = long_short_weights(sig_neutral)
ret_neutral = portfolio_returns(w_neutral, returns, transaction_cost_bps=10)

stats_neutral = performance_stats(ret_neutral)
print('=== Sector-Neutral Performance ===')
for k, v in stats_neutral.items():
    if k in ('sharpe', 'sortino', 'cagr', 'max_drawdown'):
        print(f'  {k:20s}: {v:.4f}')

## 7. Alpha Correlation Heatmap

In [ ]:
from wqalpha.visualization import plot_correlation_heatmap

# Use top 20 alphas by ICIR for the heatmap
top20 = ic_df.head(20).index.tolist()
top20_signals = all_alphas[top20]

fig = plot_correlation_heatmap(top20_signals, title='Top 20 Alpha Correlation Heatmap')
plt.show()

## Summary

The research walkthrough demonstrated:

| Step | Result |
|------|--------|
| Data loaded | 37,800 rows, 50 NSE symbols, 3 years |
| Alphas computed | All 101 WQ Formulaic Alphas |
| Best alpha (ICIR) | See output above |
| Backtest | Long-short, top/bottom decile, 10 bps cost |
| Risk | CAPM attribution + sector neutralization |

**Next steps**: Walk-forward validation, capacity analysis, Fama-French attribution.